# Student Performance Analysis
### Recreated and Modular Data Science Workflow
Based on the Extended Students Exam Scores Dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Data Ingestion & Preprocessing

In [ ]:
# Load dataset from local data folder
df = pd.read_csv('../data/Expanded_data_with_more_features.csv')
print("Shape of dataset:", df.shape)
df.head()

In [ ]:
# Drop unneeded index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)
print("Remaining columns:", df.columns.tolist())

## 2. Descriptive Statistics

In [ ]:
num_cols = ['MathScore', 'ReadingScore', 'WritingScore', 'NrSiblings']
mean = df[num_cols].mean()
median = df[num_cols].median()
mode = df[num_cols].mode().iloc[0]
std_dev = df[num_cols].std()

summary_df = pd.DataFrame({'Mean': mean, 'Median': median, 'Mode': mode, 'Std Dev': std_dev})
print(summary_df)

## 3. Correlation Analysis

In [ ]:
corr = df[['MathScore', 'ReadingScore', 'WritingScore']].corr()
print("Correlation Matrix:\n", corr)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=0.5, vmax=1.0)
plt.title("Correlation Matrix of Exam Scores")
plt.show()

## 4. Demographic Grouped Analysis

In [ ]:
# Impact of Lunch Type
lunch_group = df.groupby('LunchType')[['MathScore', 'ReadingScore', 'WritingScore']].mean()
print("Average scores by Lunch Type:\n", lunch_group)

lunch_group.plot(kind='bar', figsize=(8, 4), rot=0)
plt.title("Average Scores by Lunch Type")
plt.ylabel("Mean Score")
plt.show()

## 5. Hypothesis Testing (t-Test)
Test $H_0$: There is no significant difference in MathScore between Standard and Free/Reduced lunch.

In [ ]:
std_lunch = df[df['LunchType'] == 'standard']['MathScore'].dropna()
free_lunch = df[df['LunchType'] == 'free/reduced']['MathScore'].dropna()

t_stat, p_val = stats.ttest_ind(std_lunch, free_lunch, equal_var=False)
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.4e}")
if p_val < 0.05:
    print("Reject Null Hypothesis: Significant difference observed.")
else:
    print("Fail to Reject Null Hypothesis.")

## 6. Predictive Modeling (Linear Regression & Random Forest)

In [ ]:
# Linear Regression for MathScore
clean_df = df.dropna()
encoded = pd.get_dummies(clean_df, drop_first=True)
X = encoded.drop('MathScore', axis=1)
y = encoded['MathScore']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
reg = LinearRegression()
reg.fit(X_train, y_train)
preds = reg.predict(X_test)
print("Regression MSE:", mean_squared_error(y_test, preds))
print("Regression R²:", r2_score(y_test, preds))

In [ ]:
# Random Forest Classification for Grade (A, B, C, D, F)
bins = [0, 60, 70, 80, 90, 100]
labels = ['F', 'D', 'C', 'B', 'A']
df_clf = df.copy()
df_clf['Grade'] = pd.cut(df_clf['MathScore'], bins=bins, labels=labels)
df_clf = df_clf.dropna()

le = LabelEncoder()
for col in ['Gender', 'EthnicGroup', 'ParentEduc', 'LunchType', 'TestPrep', 'PracticeSport', 'Grade']:
    df_clf[col] = le.fit_transform(df_clf[col].astype(str))

X_clf = df_clf[['Gender', 'EthnicGroup', 'ParentEduc', 'LunchType', 'TestPrep', 'PracticeSport', 'NrSiblings']]
y_clf = df_clf['Grade']

X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_tr, y_tr)
clf_preds = clf.predict(X_te)
print("Classification Accuracy:", accuracy_score(y_te, clf_preds))
print("\nClassification Report:\n", classification_report(y_te, clf_preds))